In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fundamentals_demo

In [0]:
import requests

In [0]:
!ls

In [0]:
%sql
SELECT current_catalog();

In [0]:
!pwd

In [0]:
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
base_volume = f"/Volumes/{catalog_name}/default/fundamentals_demo"

files = ["2019.csv", "2020.csv", "2021.csv"]

for f in files:
    dbutils.fs.cp(f"file:/Workspace/Users/darsh2115@gmail.com/demo/fundamentals/{f}", f"{base_volume}/{f}")

In [0]:
df = spark.read.load(f"/Volumes/{catalog_name}/default/fundamentals_demo/*.csv", format="csv")

In [0]:
df.display()

# Define Schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", StringType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType()),
])

df = spark.read.load(f"/Volumes/demo/default/fundamentals_demo/*.csv", format="csv", schema=orderSchema)

df.display()

In [0]:
df.createOrReplaceTempView("salesOrders")
df2 = spark.sql("SELECT * FROM salesOrders")

df2.display()


In [0]:
sqlQuery = """
    SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
        SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
    FROM salesOrders \
    GROUP BY CAST(YEAR(OrderDate) AS CHAR(4))
    ORDER BY OrderYear
"""

df3 = spark.sql(sqlQuery)
df3.display()

In [0]:
from matplotlib import pyplot as plt

In [0]:
df_sales = df3.toPandas()

plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'])
plt.title('Total Revenue by Year')
plt.xlabel('Year')
plt.ylabel('Grose Revenue')